In [ ]:
!pip install python-bcb pandas matplotlib openpyxl dataframe_image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 18.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib
import seaborn as sns
from bcb import Expectativas
from datetime import datetime
import warnings
import os

import smtplib
import ssl
from email.message import EmailMessage
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.header import Header

# CONFIGURAÇÕES GERAIS
warnings.filterwarnings('ignore')
matplotlib.style.use('seaborn-v0_8-darkgrid')
matplotlib.rcParams['figure.figsize'] = (12, 7)

# CONFIG DOS E-MAILS
REMETENTE_EMAIL = "seu-email@gmail.com"
SENHA_APP = "zmsx grfa adcj uotj"
DESTINATARIOS_EMAILS = ["email1@exemplo.com", "email2@exemplo.com"]
              ] # Convertido para lista para facilitar expansão


print("INICIANDO AUTOMAÇÃO (MÚLTIPLOS DESTINATÁRIOS)")


# COLETA DE DADOS (BCB)
def coletar_dados_bcb():
    print("\n Coletando dados do Banco Central...")
    em = Expectativas()
    ep = em.get_endpoint('ExpectativasMercadoAnuais')
    data_inicio = '2025-01-01'

    mapa_indicadores = {
        'IPCA': 'IPCA',
        'Preços administrados': 'IPCA Administrados',
        'IGP-M': 'IGP-M',
        'Selic' : 'Selic',
        'Câmbio': 'Câmbio',
        'PIB Total': 'PIB Total',
        'Dívida líquida do setor público': 'Dívida líquida',
        'Conta corrente': 'Conta corrente',
    }

    try:
        df_raw = ep.query().filter(f"Data gt '{data_inicio}'").collect()
    except Exception as e:
        print(f"ERRO DE CONEXÃO COM O BCB: {e}")
        return None

    if df_raw.empty:
        return None

    df_filtrado = df_raw[df_raw['Indicador'].isin(mapa_indicadores.keys())].copy()
    df_filtrado['Indicador'] = df_filtrado['Indicador'].replace(mapa_indicadores)
    cols_existentes = [c for c in ['Indicador', 'Data', 'DataReferencia', 'Media', 'Mediana'] if c in df_filtrado.columns]
    df_filtrado = df_filtrado[cols_existentes]
    df_filtrado['Data'] = pd.to_datetime(df_filtrado['Data'])
    df_filtrado['DataReferencia'] = pd.to_numeric(df_filtrado['DataReferencia'], errors='coerce')
    anos_desejados =[2025, 2026, 2027]
    df_filtrado = df_filtrado[df_filtrado['DataReferencia'].isin(anos_desejados)]

    print(f"Dados coletados e filtrados. Total: {len(df_filtrado)}")
    return df_filtrado


# GERAÇÃO DA TABELA DE VARIAÇÃO SEMANAL
def gerar_tabela_focus(df):
    if df is None or df.empty: return None, None
    print("\nGerando Tabela Comparativa...")

    datas_unicas = sorted(df['Data'].unique())
    if not datas_unicas: return None, None
    data_recente = datas_unicas[-1]
    alvo_anterior = data_recente - pd.Timedelta(days=7)

    if alvo_anterior in datas_unicas: data_anterior = alvo_anterior
    elif len(datas_unicas) >= 2: data_anterior = datas_unicas[-2]
    else: return None, None

    data_rec_str = data_recente.strftime('%d/%m')
    data_ant_str = data_anterior.strftime('%d/%m')
    cols = ['Indicador', 'DataReferencia', 'Mediana', 'Media']
    df_rec = df[df['Data'] == data_recente][cols]
    df_ant = df[df['Data'] == data_anterior][cols]
    merged = pd.merge(df_rec, df_ant, on=['Indicador', 'DataReferencia'], how='outer', suffixes=('_rec', '_ant'))

    def processar_metrica(tipo, df_m):
        col_rec, col_ant = f'{tipo}_rec', f'{tipo}_ant'
        conds = [(df_m[col_rec] > df_m[col_ant]), (df_m[col_rec] < df_m[col_ant]), (df_m[col_rec] == df_m[col_ant])]
        df_m['Seta'] = np.select(conds, ['↑', '↓', '='], default='-')

        def fmt_rec(row):
            if pd.isna(row[col_rec]): return '-'
            return f"{str(row[col_rec]).replace('.', ',')} ({row['Seta']})"
        def fmt_ant(val):
            return '-' if pd.isna(val) else str(val).replace('.', ',')

        df_m[data_rec_str] = df_m.apply(fmt_rec, axis=1)
        df_m[data_ant_str] = df_m[col_ant].apply(fmt_ant)

        melted = pd.melt(df_m[['Indicador', 'DataReferencia', data_rec_str, data_ant_str]],
            id_vars=['Indicador', 'DataReferencia'], var_name='Data_Rel', value_name='Valor')
        melted['Metrica'] = tipo.capitalize()
        return melted

    df_mediana = processar_metrica('Mediana', merged.copy())
    df_media = processar_metrica('Media', merged.copy())

    tabela = pd.concat([df_mediana, df_media]).pivot_table(
        index=['Metrica', 'Indicador'], columns=['DataReferencia', 'Data_Rel'], values='Valor', aggfunc='first'
    ).fillna('-').sort_index(axis=1, level=[0, 1], ascending=[True, True])

    def colorir(val):
        if '↑' in val: return 'color: blue'
        if '↓' in val: return 'color: red'
        if '=' in val: return 'color: gray'
        return 'color: black'

    try: styled = tabela.style.map(colorir)
    except: styled = tabela.style.applymap(colorir)

    nome_imagem_tabela = 'tabela_para_email.png'
    try:
        import dataframe_image as dfi
        dfi.export(styled, nome_imagem_tabela, table_conversion='matplotlib')
        print(f"Imagem da tabela gerada: {nome_imagem_tabela}")
    except Exception as e:
        print(f"      Erro ao gerar imagem da tabela com dataframe_image: {e}")
        return df, None
    return df, nome_imagem_tabela


# GERAÇÃO DOS GRÁFICOS (AJUSTADO E ENCAPSULADO)
def gerar_graficos(df):
    if df is None or df.empty:
        return []

    print("\n--- Gerando Gráficos de Evolução ---")

    # Definindo os indicadores específicos solicitados
    # Nota: 'Dívida líquida' deve corresponder ao nome mapeado na função coletar_dados_bcb
    indicadores_graficos = [
        'IPCA',
        'PIB Total',
        'Câmbio',
        'Selic',
        'Dívida líquida'
    ]

    # Filtrar apenas os que existem no DataFrame
    indicadores_disponiveis = [i for i in indicadores_graficos if i in df['Indicador'].unique()]

    if not indicadores_disponiveis:
        print("Nenhum dos indicadores selecionados para gráficos foi encontrado.")
        return []

    sns.set(style="whitegrid")

    # Configurar Grid (Max 2 colunas)
    n_plots = len(indicadores_disponiveis)
    n_cols = 2
    n_rows = (n_plots + 1) // n_cols # Arredonda pra cima

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(16, 5 * n_rows))
    ax_flat = axes.flatten()

    for i, indicador in enumerate(indicadores_disponiveis):
        if i < len(ax_flat):
            ax = ax_flat[i]

            # Filtrar o DataFrame para o indicador atual E para datas >= 01/01/2025
            df_plot = df[
                (df['Indicador'] == indicador) &
                (df['Data'] >= '2025-01-01')
            ].copy()

            # Converter DataReferencia para string para garantir categorização correta no gráfico
            df_plot['DataReferencia'] = df_plot['DataReferencia'].astype(str)

            if not df_plot.empty:
                sns.lineplot(
                    data=df_plot,
                    x='Data',
                    y='Mediana',
                    hue='DataReferencia',
                    style='DataReferencia',
                    markers=False,
                    palette='tab10', # Paleta padrão vibrante
                    linewidth=2.5,
                    errorbar=None, # Remove intervalo de confiança para limpar visual
                    ax=ax
                )

                ax.set_title(f'{indicador}', fontsize=12, fontweight='bold')
                ax.set_xlabel('')
                ax.set_ylabel('')
                ax.legend(title='Ref.')

                # Melhorar a formatação da data no eixo X
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
                plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    # Remove eixos extras se houver (para não ficar quadrado branco vazio)
    for j in range(i + 1, len(ax_flat)):
        fig.delaxes(ax_flat[j])

    plt.tight_layout()

    # Salvar a imagem dos gráficos em um arquivo
    nome_arquivo_graficos = "boletim_focus_graficos.png"
    plt.savefig(nome_arquivo_graficos, dpi=100)
    plt.close(fig) # Fecha a figura para liberar memória

    print(f"Gráficos salvos em: {nome_arquivo_graficos}")
    return [nome_arquivo_graficos]


# ENVIO DO E-MAIL
def enviar_email_com_conteudo(imagem_tabela, lista_graficos):
    print("\nGerando e Enviando E-mail...")

    if not imagem_tabela or not lista_graficos:
        print("      Erro: Faltam imagens para montar o e-mail.")
        return

    msg = MIMEMultipart('related')
    msg['Subject'] = Header('Relatório Focus - Limfie', 'utf-8')
    msg['From'] = REMETENTE_EMAIL

    # Suporte para lista de emails ou string unica
    if isinstance(DESTINATARIOS_EMAILS, list):
        msg['To'] = ", ".join(DESTINATARIOS_EMAILS)
        destinatarios_str = DESTINATARIOS_EMAILS
    else:
        msg['To'] = DESTINATARIOS_EMAILS
        destinatarios_str = [DESTINATARIOS_EMAILS]

    html_body = """
    <html>
        <body style="font-family: 'Century Gothic', CenturyGothic, AppleGothic, sans-serif; color: #b7ab8d;">
            <h2 style="color: #134633;">Boletim Focus Limfie</h2>
            <p>Olá, associado(a)!</p>
            <p>Segue abaixo o Boletim semanal dos indicadores do Banco Central.</p>
            <hr>
            <h3 style="background-color: #134633; color: white; padding: 10px;">Focus Expectativas de Mercado</h3>
            <div style="text-align: center;"><img src="cid:tabela_img" style="max-width: 100%; border: 1px solid #ccc;"></div>
            <br><hr>
            <h3 style="background-color: #134633; color: white; padding: 10px;">2. Gráficos - Expectativas de Mercado</h3>
    """

    ids_graficos = []
    for i, _ in enumerate(lista_graficos):
        cid = f"grafico_{i}"
        ids_graficos.append(cid)
        html_body += f'<div style="text-align: center; margin-bottom: 20px;"><img src="cid:{cid}" style="max-width: 800px; width: 100%;"></div>'

    html_body += """<hr><p style="font-size: 12px; color: #777;">Esta mensagem é reservada à Liga de Mercado
    Financeiro da UFRJ (LIMFIE) e sua divulgação, distribuição, reprodução ou qualquer forma de uso é proibida
    e depende de prévia autorização desta instituição. O remetente utiliza o correio eletrônico no exercício do suas atividades
    da liga ou em razão dele, eximindo esta instituição de qualquer responsabilidade por utilização indevida. Se você recebeu esta mensagem
    por engano, favor eliminá-la imediatamente.</p></body></html>"""
    msg.attach(MIMEText(html_body, 'html', 'utf-8'))

    def anexar_imagem(caminho, content_id):
        try:
            with open(caminho, 'rb') as f:
                img_data = f.read()
            mime_img = MIMEImage(img_data)
            mime_img.add_header('Content-ID', f'<{content_id}>')
            mime_img.add_header('Content-Disposition', 'inline', filename=os.path.basename(caminho))
            msg.attach(mime_img)
        except Exception as e:
            print(f"      Erro ao anexar {caminho}: {e}")

    anexar_imagem(imagem_tabela, 'tabela_img')
    for arquivo, cid in zip(lista_graficos, ids_graficos): anexar_imagem(arquivo, cid)

    print("      Conectando ao servidor SMTP...")
    try:
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(REMETENTE_EMAIL, SENHA_APP)
            server.sendmail(REMETENTE_EMAIL, destinatarios_str, msg.as_bytes())

        print("\n✅ Relatório enviado com sucesso.")
    except Exception as e:
        print(f"\n❌ Erro no envio: {e}")

# ORQUESTRADOR
if __name__ == "__main__":
    if "seu.email" in REMETENTE_EMAIL:
        print("⚠️ Você esqueceu de preencher REMETENTE_EMAIL e SENHA_APP!")
    else:
        df_dados = coletar_dados_bcb()
        if df_dados is not None:
            # Gerar Tabela
            df, img_tabela = gerar_tabela_focus(df_dados)

            # Gerar Gráficos (Agora chamando a função encapsulada corretamente)
            lista_graficos = gerar_graficos(df)

            # Enviar Email
            if img_tabela and lista_graficos:
                enviar_email_com_conteudo(img_tabela, lista_graficos)
            else:
                print("Falha na geração das imagens. Email não enviado.")

INICIANDO AUTOMAÇÃO (MÚLTIPLOS DESTINATÁRIOS)

 Coletando dados do Banco Central...
Dados coletados e filtrados. Total: 10076

Gerando Tabela Comparativa...
Imagem da tabela gerada: tabela_para_email.png

--- Gerando Gráficos de Evolução ---
Gráficos salvos em: boletim_focus_graficos.png

Gerando e Enviando E-mail...
      Conectando ao servidor SMTP...

✅ Relatório enviado com sucesso.
